# Enhanced Super Resolution Generative Adversarial Network (ESRGAN)

This notebook demonstrates how to use ESRGAN for super-resolution of satellite imagery. Data is fetched from an open-access image source in S3 from the State of Indiana government captured in 2025. See the ESRGAN class documentation for more thorough details  about intent and usage, which in most cases should be a Hyperspectral or Multispectral single image band.  Separate models can be set up for individual bands, as in this example where the near infrared band is used.

The ESRGANDataProcess supports the following workflows:
1. **Training use a single image dataset input** - Imagery will be downsampled by scaling factor and downsampled inputs will be used in training.
1. **Separate target and input datasets** - Target and input datasets are provided separately, possibly from different sensor sources.  Scaling factor will be set as the factor between the two datasets' spatial resolutions.
1. **Multiple separate bands of the same scene can be provided** - Either with or without separate low-resolution input files.
1. **Single file with multiple bands can be provided** - Either with or without separate low-resolution input files.

In [ ]:
# % pip install geoai

In [ ]:
import os
from collections import OrderedDict

import numpy as np
import torch
import rasterio
from rasterio.enums import Resampling

import geoai
from geoai.esrgan import ESRGAN, ESRGANDataPreprocess, ESRGANGenerator
from geoai.inference import predict_geotiff_superres

## Download sample image dataset

In [ ]:
url = "https://gisimageryingov.s3.amazonaws.com/imageryoptimized/statewide/2025/SPW/03in/in2025_28252357_3in.tif"
if not os.path.exists("data"):
    os.makedirs("data", exist_ok=True)
download_path = geoai.download_file(
    url, output_path=os.path.join("data", os.path.basename(url))
)
download_fname = os.path.basename(url)

## Preprocessing

Generate tiles and downsample target (training/goal resolution) imagery to lower resolution for training.

In [ ]:
download_path = os.path.join("data", "in2025_28252357_3in.tif")
preprocess = ESRGANDataPreprocess(
    target_file=download_path,
    use_downsampled_targets=True,
    scale_factor=4,
    output_dir="data",
)
preprocess.initiate_preprocessing()

## Prepare tensors for input to training pipeline

These are executed as individual bands (channels) in most testing. The original reason for creating separate models for each bands is that different HSI bands may show unique patterns and properties.

In [ ]:
preprocess.to_tensor(
    save_tensor=True,
    manual_seed=37,  # Make results reproducible
    verbose=True,  # Print messages to stdout
    band=4,  # For testing, run through band 4 / NIR wavelength
    using_tiles=True,
)

## Train the Generator and Discriminator models

See ESRGAN class documentation for detailed breakdowns of methodology, key parameters and hyperparameters. 

Parameters set here include scale (needs to match factor between target and input datasets' spatial resolution), and data paths.

For training, parameters include paths to datasets, learning rate for generator and discriminator, batch size, epochs, and validation splits.

Hyperparameters for use in convolution layers (main ESRGAN and RRDB) include number of filters (patches), number of blocks (used for iteration count of the RRDB sub-model), and number of groups (can be changed for memory management).

Warmup epochs refers to the number of epochs run with only pixel and perceptual loss applied to the generator before the discriminator (which trains separately from the first epoch) is applied to generator outputs.

Lambda values for pixel, perceptual and discriminator are the factors with which each loss function is weighted when applied to the generator.  If all three values were 1, and we were not in a "warmup epoch", each loss function would be weighted .333. 

In [ ]:
esrgan = ESRGAN(
    scale=4, band=4, manual_seed=37, data_path="./data", model_path="./models"
)
esrgan.train_esrgan(
    low_res=os.path.join("data", "tensors", "y_lr_tensors.pt"),
    high_res=os.path.join("data", "tensors", "x_hr_tensors.pt"),
    lr_generator=1e-3,
    lr_discriminator=1e4,
    batch_size=16,
    num_epochs=12,
    warmup_epochs=2,
    nblocks=2,
)

## Evaluate Model

Take a completely different image (in this case, a neighboring tile from the same source dataset), and re-create downsampled tiles to test the model's ability to sharpen. 

In [ ]:
url = "https://gisimageryingov.s3.amazonaws.com/imageryoptimized/statewide/2025/SPW/03in/in2025_28252355_3in.tif"
if not os.path.exists("val"):
    os.makedirs("val", exist_ok=True)
download_path = geoai.download_file(
    url, output_path=os.path.join("val", os.path.basename(url))
)
download_fname = os.path.basename(url)

preprocess = ESRGANDataPreprocess(
    target_file=download_path,
    use_downsampled_targets=True,
    scale_factor=4,
    output_dir="val",
)
preprocess.initiate_preprocessing()
preprocess.to_tensor(
    save_tensor=True,
    manual_seed=37,  # Make results reproducible
    verbose=True,  # Print messages to stdout
    band=4,  # For testing, run through band 4 / NIR wavelength
    using_tiles=True,
)

## Inference

In [ ]:
# -----------------------------
# Config
# -----------------------------
band = 4
scale = 4

url = "https://gisimageryingov.s3.amazonaws.com/imageryoptimized/statewide/2025/SPW/03in/in2025_28252355_3in.tif"
weights_path = f"./models/{band}/best_generator.pt"

val_dir = "val"
os.makedirs(val_dir, exist_ok=True)

hr_path = geoai.download_file(
    url, output_path=os.path.join(val_dir, os.path.basename(url))
)
lr_path = os.path.join(val_dir, f"lr_x{scale}_" + os.path.basename(hr_path))
sr_out_path = os.path.join(val_dir, f"sr_x{scale}_band{band}.tif")

device = "cuda" if torch.cuda.is_available() else "cpu"

# LR tiling params (in LR pixels)
tile_size = 128
overlap = 32
batch_size = 8

# -----------------------------
# Downsample HR to LR and save to disk
# -----------------------------
with rasterio.open(hr_path) as src:
    # Read only the band you want (1-based index)
    hr = src.read(band).astype(np.float32)

    lr_h = src.height // scale
    lr_w = src.width // scale

    # Resample to LR grid
    lr = src.read(
        band,
        out_shape=(lr_h, lr_w),
        resampling=Resampling.average,  # use average for continuous imagery
    ).astype(np.float32)

    # Compute LR transform (pixel size increases by `scale`)
    lr_transform = src.transform * rasterio.Affine.scale(scale, scale)

    lr_profile = src.profile.copy()
    lr_profile.update(
        height=lr_h,
        width=lr_w,
        count=1,
        dtype="float32",
        transform=lr_transform,
        nodata=src.nodata,
        compress="lzw",
    )

with rasterio.open(lr_path, "w", **lr_profile) as dst:
    dst.write(lr, 1)

print("HR source:", hr_path)
print("LR written:", lr_path)

# -----------------------------
# Load ESRGAN generator + weights
# -----------------------------
gen = ESRGANGenerator(in_nc=1, out_nc=1, nf=64, nb=2, gc=32, scale=scale)
state_dict = torch.load(weights_path, map_location="cpu")
gen.load_state_dict(OrderedDict(state_dict))


# ------------------------
# Preprocess for inference
# ------------------------
def preprocess_fn(tile_chw: np.ndarray) -> np.ndarray:
    t = tile_chw.astype(np.float32)

    # if looks like uint8, scale
    if t.max() > 1.5:
        t = t / 255.0

    # per-tile min/max normalize
    t = t - np.nanmin(t)
    mx = np.nanmax(t)
    if mx > 0:
        t = t / mx

    return np.nan_to_num(t, nan=0.0)


# -----------------------------
# Run SR inference: LR raster to mosaicked HR raster
# -----------------------------
predict_geotiff_superres(
    model=gen,
    input_raster=lr_path,
    output_raster=sr_out_path,  # high-res inference output
    scale=scale,
    tile_size=tile_size,
    overlap=overlap,
    batch_size=batch_size,
    input_bands=[1],  # low-res file is single-band
    num_classes=1,
    output_dtype="float32",
    output_nodata=-9999.0,
    blend_mode="spline",
    blend_power=2,
    tta=False,
    preprocess_fn=preprocess_fn,
    device=device,
    compress="lzw",
    verbose=True,
)

print("SR output:", sr_out_path)

## Inference Example Outputs
Example above was run through with parameters and hyperparameters indicated.  Output model (saved as best_generator.pt zipfile) was used and provided as input to `predict_geotiff_superres` method to infer sharpened image tiles, and then merge them.  

Maps below represent the same scene. From top left and continuing clockwise, they show:
- Input imagery for inference (low-res), at 12" spatial resolution
- "Control" image, upsampled from input using GDAL and cubic convolution resampling to 3" resolution
- ESRGAN inference result, 3" resolution
- "Ground Truth" image at original 3" resolution

![esrgan_inference](../assets/esrgan_inference.png)